In [1]:

%pip install tqdm
!git clone https://github.com/NVlabs/stylegan2-ada.git

Note: you may need to restart the kernel to use updated packages.


fatal: destination path 'stylegan2-ada' already exists and is not an empty directory.


In [2]:
%pip install opencv-python


Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install --upgrade numpy


Note: you may need to restart the kernel to use updated packages.


In [4]:
#Libraries

import cv2
import os
import subprocess

In [5]:
class StyleGAN:
  def __init__(self,data_path,img_size):
    self.data_path = data_path
    self.img_size = img_size

  def data_preprocess(self,data_path):
    """
    Preprocess images: resize them to the specified size.
    """
    processed_count = 0

    for root,dirs,files in os.walk(data_path):
      for file in files:
        img_path = os.path.join(root,file)
        img = cv2.imread(img_path)

        if img is None:
          print(f"File not read: {img_path}")
          continue

        img_resized = cv2.resize(img,(self.img_size,self.img_size))
        # Save resized image back (overwrite or save to new location)
        cv2.imwrite(img_path, img_resized)
        processed_count += 1

    print(f"Processed {processed_count} images from: {data_path}")
    return processed_count

  def StyleGAN_generation(self, input_images_dir="../data/stylegan_images", output_resolution=512):
    """
    Generate images using StyleGAN2-ADA.
    
    Steps:
    1. Create TFRecords from images
    2. Train the model (or use pretrained)
    3. Generate new images
    """
    import os
    # Convert to absolute paths to avoid Windows path issues
    # Convert relative paths to absolute paths
    input_images_dir_abs = os.path.abspath(input_images_dir)
    # For tfrecords, use the same base as input_images_dir
    if input_images_dir.startswith("../"):
        # If input is ../data/stylegan_images, tfrecords should be ../data/tfrecords_dataset
        tfrecords_path = os.path.abspath(input_images_dir.replace("stylegan_images", "tfrecords_dataset"))
    else:
        # If absolute path, construct accordingly
        base_dir = os.path.dirname(input_images_dir_abs)
        tfrecords_path = os.path.join(base_dir, "tfrecords_dataset")
    
    # Create directories if they don't exist
    os.makedirs(tfrecords_path, exist_ok=True)
    
    # 1) Create TFRecords from images
    print("Step 1: Creating TFRecords from images...")
    print(f"  Input images: {input_images_dir_abs}")
    print(f"  Output TFRecords: {tfrecords_path}")
    
    result = subprocess.run(
        [
            "python", "stylegan2-ada/dataset_tool.py",
            "create_from_images",
            tfrecords_path,
            input_images_dir_abs,
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    
    if result.returncode != 0:
        print("ERROR: Failed to create TFRecords")
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
        raise subprocess.CalledProcessError(result.returncode, result.args, result.stdout, result.stderr)
    
    print("TFRecords created successfully!")
    
    # 2) Train the model (comment out if using pretrained only)
    print("Step 2: Training model...")
    results_dir = os.path.join(os.getcwd(), "results")
    os.makedirs(results_dir, exist_ok=True)
    
    result = subprocess.run(
        [
            "python", "stylegan2-ada/train.py",
            "--outdir", results_dir,
            "--snap=10",
            f"--data={tfrecords_path}",
            "--augpipe=bgcfnc",
            f"--res={output_resolution}",
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    
    if result.returncode != 0:
        print("ERROR: Training failed")
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
        raise subprocess.CalledProcessError(result.returncode, result.args, result.stdout, result.stderr)
    
    # 3) Generate images from trained network
    print("Step 3: Generating images...")
    subprocess.run(
        [
            "python", "stylegan2-ada/generate.py",
            "--outdir=./generated_images",
            "--trunc=1",
            "--seeds=0-10",
            "--network=./results/00000-stylegan2-custom-gpus1-batch4-gamma0.2048/network-snapshot-*.pkl",
        ],
        check=True,
    )


In [2]:
%pip uninstall tensorflow 

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import tensorflow as tf 
%pip install tensorflow==1.15


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow==1.15 (from versions: 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow==1.15


In [1]:
# Main execution - Run this to generate images

# Initialize StyleGAN
gan = StyleGAN(data_path="../data/stylegan_images", img_size=512)

# Generate images (this will: create TFRecords, train model, and generate images)
gan.StyleGAN_generation(input_images_dir="../data/stylegan_images", output_resolution=512)

NameError: name 'StyleGAN' is not defined

In [ ]:
# Usage example:
# 
# 1. First, make sure stylegan2-ada is cloned (run Cell 0)
# 2. Initialize the StyleGAN class
# gan = StyleGAN(data_path="../data/stylegan_images", img_size=512)
# 
# 3. Optional: Preprocess/resize images (if needed)
# gan.data_preprocess("../data/stylegan_images")
# 
# 4. Generate images (this will train and generate)
# gan.StyleGAN_generation(input_images_dir="../data/stylegan_images", output_resolution=512)
# 
# NOTE: Training StyleGAN takes a LONG time (hours/days) and requires a GPU!
# For testing, you might want to use a pretrained model first.